# GigaEvo from scratch — MAP-Elites + LLM in ~150 lines

GigaEvo is an evolutionary loop in which an **LLM proposes and mutates candidate Python
programs**, each task supplies its own **fitness function**, and a **MAP-Elites archive**
keeps the best program per *behavior cell* instead of a single global winner.

This notebook rebuilds that loop from the most basic blocks — sequential, no `gigaevo`
imports in Part 1, but using a **real problem from this repo** (the AlphaEvolve
26-circle packing benchmark) and the real GigaEvo names:

1. an LLM client (LiteLLM proxy, Qwen3-235B instruct)
2. a real task: pack 26 circles in the unit square (`problems/alphaevolve/packing_circles/n_26`)
3. the problem's frozen `validate.py`
4. a `Program` record
5. a 2-D behavior descriptor (code length × code complexity) mapping programs to behavior cells
6. a MAP-Elites archive (a dict)
7. a `MutationSuggestionStage` and a mutation operator, with simple prompts
8. the evolution loop, then results
9. posthoc: a **memory system** — `MemoryCard`s written on gain events, read back into
   mutation prompts

**Part 2** then replays the same pieces with the real `gigaevo` objects — stage DAG,
`Program`, `MapElitesIsland`, `MutationOperator` — to show what the framework adds.

Requires `OPENAI_API_KEY` (the LiteLLM proxy key) in the environment.


## 1. LLM client

Real GigaEvo routes calls through configurable LLM routers with retries and strict
structured output. The minimal version is one `openai` client pointed at the in-cluster
LiteLLM proxy (`tools/litellm.sh`, inventory in `experiments/infrastructure.yaml`), and
one helper function. The proxy IP must be in `NO_PROXY`, or the system HTTP proxy
swallows the calls.


In [ ]:
import os

from openai import OpenAI

os.environ["NO_PROXY"] = os.environ.get("NO_PROXY", "") + ",INTERNAL_IP"

MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"
client = OpenAI(
    base_url="http://localhost:8000/v1", api_key=os.environ["OPENAI_API_KEY"]
)


def llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
        max_tokens=4000,
    )
    return response.choices[0].message.content


llm("Reply with just: OK")

## 2. The task (`task_description.txt`)

Problems live in `problems/<name>/`: a `task_description.txt`, a frozen `validate.py`,
a `metrics.yaml` naming the fitness metric, and `initial_programs/`. A candidate
solution is Python source implementing **`def entrypoint():`**.

We use a real one — the AlphaEvolve circle-packing benchmark: place **26 non-overlapping
circles of variable radii inside the unit square, maximizing the sum of radii**. It is
genuinely hard (non-convex, many local optima); AlphaEvolve's published packing reaches
a sum of ≈ 2.635, which `metrics.yaml` records as the target.


In [ ]:
from pathlib import Path

import numpy as np

PROBLEM_DIR = next(
    p / "problems" / "alphaevolve" / "packing_circles" / "n_26"
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "problems" / "alphaevolve").exists()
)
TASK_DESCRIPTION = (PROBLEM_DIR / "task_description.txt").read_text()
TARGET = 2.635  # metrics.yaml upper_bound

print(TASK_DESCRIPTION[:400] + "\n[...]")

## 3. The validate function (`validate.py`)

Each problem ships a frozen `validate.py`: it raises on constraint violations
(containment, overlap, wrong shape) and returns the metrics dict, whose primary
`fitness` key is declared in `metrics.yaml`. We import the real file.

The engine runs a program's `entrypoint()` inside a sandboxed stage DAG with timeouts;
here both collapse into one `evaluate()` function with a `signal.alarm` as the poor
man's sandbox. A program that fails execution or validation ends in the *invalid*
state — but its error text is kept: §10's loop hands it back to the mutation prompt as
an explicit fix-me, the way GigaEvo surfaces stage errors to the mutator.


In [ ]:
import importlib.util
import signal

spec = importlib.util.spec_from_file_location(
    "packing_validate", PROBLEM_DIR / "validate.py"
)
validate_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate_module)
validate = validate_module.validate


def _timeout(signum, frame):
    raise TimeoutError("entrypoint() exceeded 60 seconds")


def evaluate(code: str):
    """Execute entrypoint() and validate its output. Returns (metrics, error); exactly one is None."""
    namespace = {}
    signal.signal(signal.SIGALRM, _timeout)
    signal.alarm(60)
    try:
        exec(code, namespace)
        if "entrypoint" not in namespace:
            return None, "source does not define entrypoint()"
        return validate(namespace["entrypoint"]()), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    finally:
        signal.alarm(0)

## 4. `Program` — the unit of evolution

Mirrors `gigaevo.programs.Program`: the source `code`, the `metrics` dict, a
`stage_results` dict that stages write into, and lineage (`parents`, `generation`).


In [ ]:
from dataclasses import dataclass, field
import itertools

_next_id = itertools.count()


@dataclass
class Program:
    id: int
    code: str
    metrics: dict[str, float]
    cell: tuple[int, int]
    parents: list[int]
    generation: int
    stage_results: dict[str, str] = field(default_factory=dict)

    @property
    def fitness(self) -> float:
        return self.metrics["fitness"]

## 5. Behavior descriptors → behavior cell

Fitness alone collapses the population to a single winner. MAP-Elites adds *behavior
descriptors*: measurable traits of a solution that map each program to a **behavior
cell**, and the archive keeps the best program per cell. That preserves diverse
approaches — a short constructive formula and a long optimization loop both stay alive
as mutation material.

We use two descriptors, so the archive becomes a 2-D grid:

- **code length** — characters of source, 12 cells of 500 spanning 2000–8000:
  LLM-evolved packings routinely run several thousand characters, so the axis covers
  that band (shorter programs land in the first cell, longer in the last);
- **code complexity** — how much *machinery* the code uses: the number of calls,
  arithmetic ops, loops and branches in the AST, 5 cells of 25. This is a miniature of
  GigaEvo's real `ComputeComplexityStage`, which scores a weighted version of exactly
  these node counts.

The two are deliberately correlated-but-different: a long program full of data tables
is not complex; a short one with nested optimization loops is.


In [ ]:
import ast

N_LENGTH_CELLS = 12
LENGTH_OFFSET = 2000  # evolved programs are usually 2000+ chars
LENGTH_CELL_WIDTH = 500  # characters of source code per cell -> axis spans 2000-8000
N_COMPLEXITY_CELLS = 5
COMPLEXITY_CELL_WIDTH = 25  # AST ops per cell


def code_complexity(code: str) -> int:
    tree = ast.parse(code)
    return sum(
        isinstance(n, (ast.Call, ast.BinOp, ast.For, ast.While, ast.If))
        for n in ast.walk(tree)
    )


def behavior_cell(code: str) -> tuple[int, int]:
    return (
        min(max(len(code) - LENGTH_OFFSET, 0) // LENGTH_CELL_WIDTH, N_LENGTH_CELLS - 1),
        min(code_complexity(code) // COMPLEXITY_CELL_WIDTH, N_COMPLEXITY_CELLS - 1),
    )

## 6. The MAP-Elites archive

The heart of the algorithm — in real GigaEvo a Redis-backed `MapElitesIsland`, here just
a dict: one elite per behavior cell, replaced whenever a newcomer in the same cell scores
higher.


In [ ]:
archive: dict[tuple[int, int], Program] = {}


def try_insert(program: Program) -> bool:
    incumbent = archive.get(program.cell)
    if incumbent is None or program.fitness > incumbent.fitness:
        archive[program.cell] = program
        return True
    return False

## 7. `MutationSuggestionStage`

After a program is evaluated, real GigaEvo runs `MutationSuggestionStage`: an LLM analyst
reads the program's code, metrics, lineage cards and memory, and writes *actionable
suggestions* into `stage_results` for the mutator to consume. The minimal version is one
LLM call producing a short suggestion string. We run it only for programs that enter the
archive — only elites ever become parents.


In [ ]:
SUGGESTION_PROMPT = """You are analyzing a program from an evolutionary code-optimization run.

## Task
{task}

## Program (fitness: {fitness:.4f}, target: {target})
```python
{code}
```

In 2-3 sentences, give ONE concrete, actionable suggestion for how a mutation
could improve this program's fitness."""


def mutation_suggestion_stage(program: Program) -> None:
    prompt = SUGGESTION_PROMPT.format(
        task=TASK_DESCRIPTION, fitness=program.fitness, target=TARGET, code=program.code
    )
    program.stage_results["mutation_suggestion"] = llm(prompt)

## 8. Initial program (`initial_programs/`)

Real problems seed the archive from `initial_programs/*.py`. We take the problem's own
`grid.py` — a deliberately weak 5×6 grid of small circles, leaving plenty of room to
improve.


In [ ]:
INITIAL_PROGRAM = (PROBLEM_DIR / "initial_programs" / "grid.py").read_text()

metrics, error = evaluate(INITIAL_PROGRAM)
assert error is None, error
seed = Program(
    id=next(_next_id),
    code=INITIAL_PROGRAM,
    metrics=metrics,
    cell=behavior_cell(INITIAL_PROGRAM),
    parents=[],
    generation=0,
)
try_insert(seed)
mutation_suggestion_stage(seed)
print(f"seed fitness = {seed.fitness:.4f} / target {TARGET}   cell = {seed.cell}")
print(f"suggestion: {seed.stage_results['mutation_suggestion']}")

## 9. The mutation operator

The LLM **is** the mutation operator (a `MutationOperator` in real GigaEvo, fed by a
`MutationContext` built from multiple parents, insights and memory, parsed with strict
structured output). The minimal version: one parent, one context block, and the reply's
first fenced code block. The context block is the analyst's suggestion — or, when the
parent failed validation, **the validator's error rendered as an explicit fix-me**.
GigaEvo does the same: stage errors are part of the mutation context, so the LLM sees
*why* a program is invalid, not just that it is.


In [ ]:
MUTATION_PROMPT = """You are mutating a Python program in an evolutionary code-optimization run.

## Task
{task}

## Parent program (fitness: {fitness:.4f}, target: {target})
```python
{code}
```

{context}

Write an improved version. You may follow the guidance or change the approach entirely.
Reply with ONLY one Python code block containing the complete new program."""


def extract_code(reply: str) -> str | None:
    if reply.count("```") < 2:
        return None
    block = reply.split("```")[1]
    return block.removeprefix("python").strip() or None


def mutation_context(parent: Program) -> str:
    if "error" in parent.stage_results:
        return (
            "## Validator error — this program is INVALID, the mutation MUST fix it\n"
            + parent.stage_results["error"]
        )
    return "## Suggestion from analysis\n" + parent.stage_results["mutation_suggestion"]


def mutation_operator(parent: Program) -> str | None:
    prompt = MUTATION_PROMPT.format(
        task=TASK_DESCRIPTION,
        fitness=parent.fitness,
        target=TARGET,
        code=parent.code,
        context=mutation_context(parent),
    )
    return extract_code(llm(prompt))

## 10. The evolution loop (`run.py`)

The real entry point is `python run.py problem.name=<name>` — an asynchronous engine
doing exactly this: select a parent elite, run the mutation operator, evaluate the child
through the stage DAG, insert into the archive. Run sequentially it is two nested loops;
each new elite also gets its `MutationSuggestionStage` pass.

Invalid mutants are not silently dropped: the failed program keeps GigaEvo's sentinel
fitness (-1000), the validator error lands in its `stage_results`, and the mutator gets
one **repair attempt** with that error in the prompt (§9's error context path).

10 generations × 3 mutants ≈ 30 mutation calls plus suggestion and repair calls — a few
minutes against the proxy.


In [ ]:
import random

N_GENERATIONS = 10
MUTANTS_PER_GENERATION = 3

rng = random.Random(0)
best_per_generation = []

for generation in range(1, N_GENERATIONS + 1):
    for _ in range(MUTANTS_PER_GENERATION):
        parent = rng.choice(list(archive.values()))
        code = mutation_operator(parent)
        if code is None:
            print(f"  gen {generation}: reply had no code block — skipped")
            continue
        metrics, error = evaluate(code)
        if error is not None:
            invalid = Program(
                id=next(_next_id),
                code=code,
                metrics={"fitness": -1000.0},
                cell=(-1, -1),
                parents=[parent.id],
                generation=generation,
                stage_results={"error": error},
            )
            print(
                f"  gen {generation}: invalid mutant ({error.splitlines()[0][:100]}) — repair attempt"
            )
            code = mutation_operator(invalid)
            if code is None:
                print(f"  gen {generation}: repair reply had no code block — discarded")
                continue
            metrics, error = evaluate(code)
            if error is not None:
                print(
                    f"  gen {generation}: repair still invalid ({error.splitlines()[0][:100]}) — discarded"
                )
                continue
            parent = invalid
        child = Program(
            id=next(_next_id),
            code=code,
            metrics=metrics,
            cell=behavior_cell(code),
            parents=[parent.id],
            generation=generation,
        )
        inserted = try_insert(child)
        if inserted:
            mutation_suggestion_stage(child)
        verdict = "new elite" if inserted else "rejected"
        print(
            f"  gen {generation}: {parent.id} ({parent.fitness:.4f}, cell {parent.cell}) -> {child.id} ({child.fitness:.4f}, cell {child.cell})  {verdict}"
        )
    best = max(p.fitness for p in archive.values())
    best_per_generation.append(best)
    print(
        f"generation {generation:2d} | best {best:.4f} / {TARGET} | cells filled: {len(archive)}/{N_LENGTH_CELLS * N_COMPLEXITY_CELLS}"
    )

## 11. Results

Left: best fitness in the archive per generation. Right: the **2-D MAP-Elites grid** —
one elite per (code length × code complexity) cell, colored by fitness; empty cells were
never reached. Below: the best packing found, drawn in the unit square.


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(range(1, len(best_per_generation) + 1), best_per_generation, marker="o")
ax1.axhline(TARGET, ls="--", c="gray", label=f"target {TARGET}")
ax1.set(
    xlabel="generation",
    ylabel="best fitness (sum of radii)",
    title="Best fitness in archive",
)
ax1.legend()

grid = np.full((N_COMPLEXITY_CELLS, N_LENGTH_CELLS), np.nan)
for (length_cell, complexity_cell), program in archive.items():
    grid[complexity_cell, length_cell] = program.fitness
im = ax2.imshow(
    grid, origin="lower", cmap="viridis", vmin=0, vmax=TARGET, aspect="auto"
)
for (length_cell, complexity_cell), program in archive.items():
    ax2.text(
        length_cell,
        complexity_cell,
        f"{program.fitness:.2f}",
        ha="center",
        va="center",
        color="white",
        fontsize=8,
    )
ax2.set_xticks(
    range(N_LENGTH_CELLS),
    [f"{LENGTH_OFFSET + c * LENGTH_CELL_WIDTH}+" for c in range(N_LENGTH_CELLS)],
    fontsize=8,
)
ax2.set_yticks(
    range(N_COMPLEXITY_CELLS),
    [f"{c * COMPLEXITY_CELL_WIDTH}+" for c in range(N_COMPLEXITY_CELLS)],
)
ax2.set(
    xlabel="code length (chars)",
    ylabel="code complexity (AST ops)",
    title="MAP-Elites archive",
)
fig.colorbar(im, ax=ax2, label="elite fitness")

plt.tight_layout()
plt.show()

In [ ]:
best = max(archive.values(), key=lambda p: p.fitness)

namespace = {}
exec(best.code, namespace)
packing = np.asarray(namespace["entrypoint"](), dtype=float)

fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, ec="gray", lw=1.5))
for x, y, r in packing:
    ax.add_patch(plt.Circle((x, y), r, fc="tab:blue", alpha=0.4, ec="tab:blue"))
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect("equal")
ax.set_title(
    f"Best packing: sum of radii {best.fitness:.4f} / {TARGET} (generation {best.generation})"
)
plt.show()

In [ ]:
print(
    f"best fitness {best.fitness:.4f} / target {TARGET}  (program {best.id}, parents {best.parents}, generation {best.generation})"
)
print()
print(best.code)

---
# Part 1b — memory, posthoc

Evolution as run above is amnesiac: every mutation starts from the parent's code and a
fresh suggestion, and ideas that worked must be rediscovered over and over. GigaEvo's
memory system fixes this with two independent halves (separate config knobs — a writer
and a reader):

- **writer** — when a child *beats its parent* (a **gain event**), an LLM distills what
  made the difference into a **`MemoryCard`**: a short transferable idea, stamped with
  its origin program and fitness gain;
- **reader** — when mutating, cards are selected from memory and injected into the
  mutation prompt, and each card's **reputation** (uses / wins) is updated by whether
  the mutant improved on its parent.

The real reader scores cards by reputation and runs an auction over them; here we do
the simplest instructive thing — **read a random card** — and keep evolving the Part 1
archive.


## 12. `MemoryCard` and the writer

A card records the idea, where it came from, how large the gain was, and its usage
stats. The writer fires on gain events. To start with a non-empty memory we harvest one
card posthoc from Part 1's largest gain — seed → best.


In [ ]:
@dataclass
class MemoryCard:
    id: int
    insight: str
    origin: int  # id of the child program whose gain produced this card
    gain: float  # fitness delta over the parent
    uses: int = 0
    wins: int = 0


memory: list[MemoryCard] = []

CARD_WRITE_PROMPT = """An evolutionary code-optimization run just produced a gain event:
the child program below beat its parent by {gain:.4f} fitness.

## Task
{task}

## Parent (fitness {parent_fitness:.4f})
```python
{parent_code}
```

## Child (fitness {child_fitness:.4f})
```python
{child_code}
```

In 1-2 sentences, distill the ONE transferable idea that made the child better.
Phrase it as advice applicable to OTHER programs for this task, not as a description
of this particular diff."""


def write_memory_card(parent: Program, child: Program) -> MemoryCard:
    insight = llm(
        CARD_WRITE_PROMPT.format(
            gain=child.fitness - parent.fitness,
            task=TASK_DESCRIPTION,
            parent_fitness=parent.fitness,
            parent_code=parent.code,
            child_fitness=child.fitness,
            child_code=child.code,
        )
    )
    card = MemoryCard(
        id=len(memory),
        insight=insight,
        origin=child.id,
        gain=child.fitness - parent.fitness,
    )
    memory.append(card)
    return card


bootstrap_card = write_memory_card(seed, best)
print(
    f"card {bootstrap_card.id} (gain +{bootstrap_card.gain:.4f}): {bootstrap_card.insight}"
)

## 13. The reader — a random card into the mutation prompt

The reader picks a card and the mutation prompt grows an `## Idea from memory` block —
that is the whole difference from §9. `rng.choice` is the zero-intelligence baseline
reader; the real one is reputation-weighted.


In [ ]:
MEMORY_MUTATION_PROMPT = MUTATION_PROMPT.replace(
    "Write an improved version.",
    """## Idea from memory (distilled from an earlier successful mutation)
{memory_insight}

Write an improved version.""",
)


def read_memory_card() -> MemoryCard:
    return rng.choice(memory)


def mutation_operator_with_memory(parent: Program, card: MemoryCard) -> str | None:
    prompt = MEMORY_MUTATION_PROMPT.format(
        task=TASK_DESCRIPTION,
        fitness=parent.fitness,
        target=TARGET,
        code=parent.code,
        context=mutation_context(parent),
        memory_insight=card.insight,
    )
    return extract_code(llm(prompt))

## 14. Evolution with memory

The §10 loop again, continuing on the same archive, with the two memory halves wired
in: read a card before each mutation (a *use*); when a child beats its parent, count a
*win* for the card that was in the prompt and write a new card from the gain event.
Reputation accumulates on the cards — in the real system that is the signal the auction
reader exploits.


In [ ]:
MEMORY_GENERATIONS = 3

for generation in range(N_GENERATIONS + 1, N_GENERATIONS + MEMORY_GENERATIONS + 1):
    for _ in range(MUTANTS_PER_GENERATION):
        parent = rng.choice(list(archive.values()))
        card = read_memory_card()
        card.uses += 1
        code = mutation_operator_with_memory(parent, card)
        if code is None:
            print(f"  gen {generation}: reply had no code block — skipped")
            continue
        metrics, error = evaluate(code)
        if error is not None:
            print(f"  gen {generation}: invalid mutant ({error}) — discarded")
            continue
        child = Program(
            id=next(_next_id),
            code=code,
            metrics=metrics,
            cell=behavior_cell(code),
            parents=[parent.id],
            generation=generation,
        )
        inserted = try_insert(child)
        if inserted:
            mutation_suggestion_stage(child)
        if child.fitness > parent.fitness:
            card.wins += 1
            new_card = write_memory_card(parent, child)
            print(
                f"  gen {generation}: GAIN EVENT +{new_card.gain:.4f} -> card {new_card.id} written"
            )
        verdict = "new elite" if inserted else "rejected"
        print(
            f"  gen {generation}: card {card.id} + parent {parent.id} ({parent.fitness:.4f}) -> {child.id} ({child.fitness:.4f})  {verdict}"
        )
    best_fitness = max(p.fitness for p in archive.values())
    best_per_generation.append(best_fitness)
    print(
        f"generation {generation:2d} | best {best_fitness:.4f} / {TARGET} | memory: {len(memory)} cards"
    )

The cards now carry both content and reputation. In real GigaEvo this is where the loop
closes: uses and wins feed each card's posterior, the auction reader prefers cards that
actually helped, and stale ideas fade.


In [ ]:
print(f"memory after {MEMORY_GENERATIONS} generations: {len(memory)} cards")
print()
for card in memory:
    print(f"card {card.id}  gain=+{card.gain:.4f}  uses={card.uses}  wins={card.wins}")
    print(f"  {card.insight}")
    print()

best = max(archive.values(), key=lambda p: p.fitness)
print(f"best after memory generations: {best.fitness:.4f} / {TARGET}")

---
# Part 2 — the same loop with real GigaEvo objects

Part 1 fit in ~150 lines because it skipped the plumbing. Now we rebuild the very same
loop, on the very same problem, out of the production objects — the ones `run.py`
normally assembles from Hydra config. Hand-wiring them (the pattern of
`tests/integration/test_mini_run.py`, the repo's "children's version of run.py") shows
exactly which real object each Part 1 block corresponds to.

Three practical notes: the whole stack is asyncio-native (Jupyter's event loop lets us
`await` right in cells); instead of the production Redis backends we use the disk
backends — no server needed; and we silence `loguru`, because the DAG (deliberately)
logs every invalid program with a full traceback — informative in production, a wall of
red in a notebook. The failures still show up as data: sentinel fitness and the stage's
`error` field.


In [ ]:
import sys
import tempfile

from langchain_openai import ChatOpenAI
from loguru import logger

from gigaevo.database.disk_program_storage import (
    DiskProgramStorage,
    DiskProgramStorageConfig,
)
from gigaevo.database.state_manager import ProgramStateManager
from gigaevo.entrypoint.evolution_context import EvolutionContext
from gigaevo.entrypoint.lineage_memory_pipeline import IntraMemoryPipelineBuilder
from gigaevo.evolution.mutation.mutation_operator import LLMMutationOperator
from gigaevo.evolution.storage.archive_storage import DiskArchiveStorageFactory
from gigaevo.evolution.strategies.elite_selectors import ScalarTournamentEliteSelector
from gigaevo.evolution.strategies.island import IslandConfig
from gigaevo.evolution.strategies.migrant_selectors import RandomMigrantSelector
from gigaevo.evolution.strategies.models import BehaviorSpace, LinearBinning
from gigaevo.evolution.strategies.multi_island import MapElitesMultiIsland
from gigaevo.evolution.strategies.removers import FitnessArchiveRemover
from gigaevo.evolution.strategies.selectors import SumArchiveSelector
from gigaevo.llm.models import MultiModelRouter
from gigaevo.problems.context import ProblemContext
from gigaevo.problems.initial_loaders import DirectoryProgramLoader
from gigaevo.programs.program import Program as GigaProgram
from gigaevo.utils.trackers.backends.disk import DiskMetricsBackend
from gigaevo.utils.trackers.configs import DiskMetricsConfig
from gigaevo.utils.trackers.core import GenericLogger

logger.remove()
logger.add(sys.stderr, level="CRITICAL")

## 15. `ProblemContext`, storage, telemetry

`ProblemContext` wraps the problem directory: it reads `task_description.txt` and parses
`metrics.yaml` into a `MetricsContext` (which metric is primary, bounds, direction).
`DiskProgramStorage` is the Redis-free `ProgramStorage` backend — every `Program` is
persisted as JSON. The `GenericLogger` writer is the telemetry sink every stage reports
into (production runs point it at Redis/TensorBoard/W&B).


In [ ]:
problem_ctx = ProblemContext(PROBLEM_DIR)
metrics_context = problem_ctx.metrics_context

RUN_DIR = Path(tempfile.mkdtemp(prefix="gigaevo_nb_"))
storage = DiskProgramStorage(
    DiskProgramStorageConfig(root_dir=str(RUN_DIR / "programs"), key_prefix="n26")
)
writer = GenericLogger(
    DiskMetricsBackend(DiskMetricsConfig(root_dir=str(RUN_DIR / "metrics")))
)

print("primary metric:", metrics_context.get_primary_key())
print("run dir:", RUN_DIR)

## 16. The LLM: `MultiModelRouter`

Part 1's `llm()` helper becomes a `MultiModelRouter`: a list of LangChain `ChatOpenAI`
models sampled by probability, with retries, concurrency limits, and structured-output
handling. It verifies the endpoint is alive at construction time.


In [ ]:
chat_model = ChatOpenAI(
    model=MODEL,
    base_url="http://localhost:8000/v1",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.8,
    max_tokens=4000,
    request_timeout=600,
    max_retries=2,
)
router = MultiModelRouter(
    models=[chat_model],
    probabilities=[1.0],
    name="mutation",
    structured_output_method="auto",
)

## 17. The stage DAG

Part 1's `evaluate()` becomes a **DAG of stages** built by a pipeline builder. Executing
`entrypoint()` happens in a subprocess (`CallProgramFunction`), the frozen validator runs
in `CallValidatorFunction`, metrics get merged and checked, and — same as our Part 1
stage — `MutationSuggestionStage` produces insights that flow into `MutationContextStage`
(whose output the mutator later renders into its prompt). Each program gets a fresh DAG
built from the blueprint; stages run concurrently where dependencies allow.


In [ ]:
evo_ctx = EvolutionContext(problem_ctx=problem_ctx, llm_wrapper=router, storage=storage)
blueprint = IntraMemoryPipelineBuilder(evo_ctx, stage_timeout=300.0).build_blueprint()

print("stages:", ", ".join(blueprint.nodes))
print()
print("data-flow edges:")
for edge in blueprint.data_flow_edges:
    print(f"  {edge.source_stage} -> {edge.destination_stage}.{edge.input_name}")

The same blueprint drawn as a graph — nodes are stages, arrows are data-flow edges
labeled with the input they feed (the runner itself uses `networkx` to check the
blueprint is acyclic). Layers are topological generations: everything in one layer may
run concurrently once its inputs exist.


In [ ]:
import networkx as nx

G = nx.DiGraph()
G.add_nodes_from(blueprint.nodes)
for edge in blueprint.data_flow_edges:
    G.add_edge(edge.source_stage, edge.destination_stage, label=edge.input_name)

for layer, stage_names in enumerate(nx.topological_generations(G)):
    for name in stage_names:
        G.nodes[name]["layer"] = layer

pos = nx.multipartite_layout(G, subset_key="layer")
fig, ax = plt.subplots(figsize=(16, 8))
nx.draw_networkx_edges(
    G,
    pos,
    arrows=True,
    arrowsize=14,
    node_size=2500,
    connectionstyle="arc3,rad=0.1",
    edge_color="gray",
    ax=ax,
)
nx.draw_networkx_labels(
    G,
    pos,
    font_size=8,
    bbox=dict(boxstyle="round,pad=0.3", fc="#dceeff", ec="#4a90d9"),
    ax=ax,
)
nx.draw_networkx_edge_labels(
    G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=6, ax=ax
)
ax.set_title("Stage DAG from the blueprint")
ax.axis("off")
plt.tight_layout()
plt.show()

## 18. The real archive: `MapElitesMultiIsland`

Part 1's dict becomes an island (or several) with a configurable `BehaviorSpace`,
archive-storage backend, size limits, and pluggable selectors. Note the default
single-island setup from `config/algorithm/` maps programs by **fitness itself** —
150 linear bins over `[0, 2.635]` — rather than our Part 1 descriptors; elites are then
drawn by tournament selection instead of uniformly.

To get Part 1's 2-D length × complexity space here, a problem declares those
descriptors in its `metrics.yaml` (the DAG already computes them — that is
`ComputeComplexityStage`, and `EnsureMetricsStage` keeps whatever the manifest
declares) and lists both keys in `bins`.


In [ ]:
island_config = IslandConfig(
    island_id="fitness_island",
    behavior_space=BehaviorSpace(
        bins={
            "fitness": LinearBinning(
                min_val=0.0, max_val=2.635, num_bins=150, type="linear"
            )
        }
    ),
    max_size=75,
    archive_selector=SumArchiveSelector(fitness_keys=["fitness"]),
    archive_remover=FitnessArchiveRemover(
        fitness_key="fitness", fitness_key_higher_is_better=True
    ),
    elite_selector=ScalarTournamentEliteSelector(
        fitness_key="fitness", fitness_key_higher_is_better=True, tournament_size=99
    ),
    migrant_selector=RandomMigrantSelector(),
)
strategy = MapElitesMultiIsland(
    island_configs=[island_config],
    program_storage=storage,
    archive_storage_factory=DiskArchiveStorageFactory(storage),
)

## 19. Seeds through the DAG

`DirectoryProgramLoader` loads every file in `initial_programs/` as a `Program`. We push
each through the DAG, then hand the valid ones to the archive — the engine's acceptor
does this same filtering in production. Watch `random.py`: its circles can overlap, the
frozen validator raises, and the program gets the sentinel fitness (-1000) instead of
metrics — the *invalid* path from Part 1, for real this time.


In [ ]:
state_manager = ProgramStateManager(storage)


async def evaluate_with_dag(program: GigaProgram) -> bool:
    dag = blueprint.build(state_manager, writer=writer)
    await dag.run(program)
    return program.metrics.get("is_valid") == 1


seeds = await DirectoryProgramLoader(PROBLEM_DIR).load(storage)
for seed_program in seeds:
    valid = await evaluate_with_dag(seed_program)
    if valid:
        await strategy.add(seed_program)
    print(
        f"seed {seed_program.short_id}: fitness={seed_program.metrics['fitness']:.4f} valid={valid}"
    )

invalid_seed = next(p for p in seeds if p.metrics["is_valid"] != 1)
print()
print("what the validator stage recorded for the invalid seed:")
print(invalid_seed.stage_results["CallValidatorFunction"].error)

The real `MutationSuggestionStage` output now sits in `stage_results` — structured
insights rather than Part 1's single suggestion string. On this backend, though, it comes
back **empty**, and the reason is worth seeing once: the router auto-negotiates a
structured-output method (`json_schema` → `function_calling` → `json_mode`), and
constrained JSON-schema decoding on this vLLM *Instruct* deployment collapses the
suggester to the minimal schema-valid object `{"insights": []}` — 9 completion tokens,
no error anywhere. The insights schema *permits* an empty list, so constrained decoding
has a shortest-valid-path escape hatch (the mutation schema in §20 has required fields,
which is why mutations are unaffected). The cell below proves the prompt is fine: it
rebuilds the stage's exact prompt and sends it **unconstrained** — same model, rich
slate of suggestions. Production memory runs point this stage at the Thinking model
pool. Moral: when a structured-output stage goes quiet, diff constrained vs.
unconstrained before blaming the prompt.


In [ ]:
from gigaevo.llm.agents.factories import create_mutation_suggestion_agent

elite = next(p for p in seeds if p.metrics["is_valid"] == 1)
suggestions = elite.stage_results["MutationSuggestionStage"].output
print("stage output (constrained):", suggestions.model_dump_json())

agent = create_mutation_suggestion_agent(
    router, problem_ctx.task_description, problem_ctx.metrics_context, 5
)
state = agent.build_prompt(
    {
        "program": elite,
        "intra_card": None,
        "memory_cards": None,
        "ancestral_trail": [],
        "evolutionary_statistics": None,
        "mutation_mode": None,
    }
)
raw = await chat_model.ainvoke(state["messages"])
print()
print("same prompt, unconstrained:")
print(raw.content[:1500])

## 20. Real mutations: `LLMMutationOperator`

`mutate_single` renders the parent (code, metrics, insights from the mutation context)
into the production mutation prompt, calls the router with strict structured output, and
returns a `MutationSpec`; `Program.from_mutation_spec` turns it into a child with correct
lineage. The loop below is the sequential skeleton of what `SteadyStateEvolutionEngine`
runs asynchronously.


In [ ]:
real_mutation_operator = LLMMutationOperator(
    llm_wrapper=router, mutation_mode="rewrite", problem_context=problem_ctx
)

for step in range(1, 6):
    parents = await strategy.select_elites(1)
    spec = await real_mutation_operator.mutate_single(parents)
    if spec is None:
        print(f"step {step}: mutation produced nothing")
        continue
    child = GigaProgram.from_mutation_spec(spec)
    await storage.add(child)
    valid = await evaluate_with_dag(child)
    added = valid and await strategy.add(child)
    print(
        f"step {step}: parent {parents[0].short_id} ({parents[0].metrics['fitness']:.4f})"
        f" -> child {child.short_id} fitness={child.metrics.get('fitness', float('nan')):.4f} valid={valid} added={bool(added)}"
    )

## 21. Results from the real archive


In [ ]:
elites = sorted(
    await strategy.islands["fitness_island"].get_elites(),
    key=lambda p: p.metrics["fitness"],
    reverse=True,
)
for e in elites:
    print(
        f"{e.short_id}  fitness={e.metrics['fitness']:.4f}  generation={e.lineage.generation}"
    )

best_real = elites[0]
namespace = {}
exec(best_real.code, namespace)
packing = np.asarray(namespace["entrypoint"](), dtype=float)

fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, ec="gray", lw=1.5))
for x, y, r in packing:
    ax.add_patch(plt.Circle((x, y), r, fc="tab:green", alpha=0.4, ec="tab:green"))
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect("equal")
ax.set_title(
    f"Part 2 best packing (real gigaevo): {best_real.metrics['fitness']:.4f} / {TARGET}"
)
plt.show()

await storage.close()
writer.close()

## 22. The real engine, live: `DagRunner` + `SteadyStateEvolutionEngine`

Sections 15–21 drove everything by hand. Production replaces our `for` loops with two
cooperating async services — and both run fine inside Jupyter:

- **`DagRunner`** polls storage for `QUEUED` programs and runs many stage DAGs
  concurrently (our `evaluate_with_dag`, parallelized);
- **`SteadyStateEvolutionEngine`** runs a *dispatcher* loop (select elites → mutate via
  LLM → enqueue mutants) and an *ingestor* loop (accept finished programs into the
  archive), throttled by a backpressure model, with pluggable stoppers and periodic
  snapshots so a crashed run resumes where it left off.

We wire a fresh run directory (§21 closed the manual section's storage) and mirror
`run.py`'s startup order exactly: load seeds (they land `QUEUED`), bootstrap the
engine's iteration counter, start the DAG runner, then the engine. The engine's own
"Phase 0" evaluates and ingests the seeds — no manual `strategy.add` this time — and
`MaxMutantsStopper(8)` ends the run after eight mutants.

One gate we must wire explicitly: the **`program_acceptor`**. The engine-config default
(`DefaultProgramEvolutionAcceptor`) only checks that a program finished the DAG and has
*some* metrics — so the invalid `random.py` seed (fitness −1000, `is_valid` 0) sails into
the archive. With only two elites and two parents per mutation, that garbage program then
lands in **every** mutation prompt, children come back degenerate, and the run flatlines
below the seed. Production configs instead wire `StandardEvolutionAcceptor`, which adds a
validity gate (`is_valid > 0`), a check that every behavior key the islands bin on is
present, and a mutation-context check — the same filtering we did by hand in §19. Its
`required_behavior_keys` come from the island configs (here just `{"fitness"}`).
Production also enumerates parent pairs with `AllCombinationsParentSelector`; we keep the
simpler `RandomParentSelector`, which behaves the same with this few elites.


In [ ]:
import asyncio
import time

from gigaevo.evolution.engine.acceptor import StandardEvolutionAcceptor
from gigaevo.evolution.engine.config import SteadyStateEngineConfig
from gigaevo.evolution.engine.steady_state import SteadyStateEvolutionEngine
from gigaevo.evolution.engine.stopper import MaxMutantsStopper
from gigaevo.evolution.mutation.parent_selector import RandomParentSelector
from gigaevo.runner.dag_runner import DagRunner, DagRunnerConfig
from gigaevo.utils.metrics_tracker import MetricsTracker

ENGINE_DIR = Path(tempfile.mkdtemp(prefix="gigaevo_nb_engine_"))
engine_storage = DiskProgramStorage(
    DiskProgramStorageConfig(root_dir=str(ENGINE_DIR / "programs"), key_prefix="n26")
)
engine_writer = GenericLogger(
    DiskMetricsBackend(DiskMetricsConfig(root_dir=str(ENGINE_DIR / "metrics")))
)

engine_ctx = EvolutionContext(
    problem_ctx=problem_ctx, llm_wrapper=router, storage=engine_storage
)
engine_blueprint = IntraMemoryPipelineBuilder(
    engine_ctx, stage_timeout=300.0
).build_blueprint()
engine_strategy = MapElitesMultiIsland(
    island_configs=[island_config],
    program_storage=engine_storage,
    archive_storage_factory=DiskArchiveStorageFactory(engine_storage),
)

dag_runner = DagRunner(
    storage=engine_storage,
    dag_blueprint=engine_blueprint,
    config=DagRunnerConfig(poll_interval=0.5, max_concurrent_dags=4),
    writer=engine_writer,
)
engine = SteadyStateEvolutionEngine(
    storage=engine_storage,
    strategy=engine_strategy,
    mutation_operator=real_mutation_operator,
    config=SteadyStateEngineConfig(
        max_in_flight=3,
        parent_selector=RandomParentSelector(num_parents=2),
        program_acceptor=StandardEvolutionAcceptor(required_behavior_keys={"fitness"}),
        stopper=MaxMutantsStopper(max_mutants=8),
    ),
    writer=engine_writer,
    metrics_tracker=MetricsTracker(
        storage=engine_storage, metrics_context=metrics_context, writer=engine_writer
    ),
)

engine_seeds = await DirectoryProgramLoader(PROBLEM_DIR).load(engine_storage)
engine.metrics.iteration = len(engine_seeds)

dag_runner.start()
engine.start()
print(f"engine live: {len(engine_seeds)} seeds queued, stopping after 8 mutants")

The engine now runs in the background of Jupyter's event loop. Since we silenced its
logs, the loop below is our progress display: every 15 s it prints the engine's own
counters and the current archive best, until the stopper ends the run. Watch
`rej_valid` — the acceptor rejecting the invalid `random.py` seed (and any invalid
mutants) is that counter ticking up.


In [ ]:
t0 = time.time()
while not engine.task.done():
    await asyncio.sleep(15)
    elites = await engine_strategy.islands["fitness_island"].get_elites()
    best = max((p.metrics["fitness"] for p in elites), default=float("nan"))
    print(
        f"t={time.time() - t0:4.0f}s  mutants={engine.metrics.mutations_created}"
        f"  ingested={engine.metrics.programs_processed}  added={engine.metrics.added}"
        f"  rej_valid={engine.metrics.rejected_validation}"
        f"  archive={len(elites)}  best={best:.4f}"
    )

await engine.stop()
await dag_runner.stop()

engine_elites = sorted(
    await engine_strategy.islands["fitness_island"].get_elites(),
    key=lambda p: p.metrics["fitness"],
    reverse=True,
)
print()
print(
    f"engine run done: {engine.metrics.mutations_created} mutants dispatched,"
    f" {engine.metrics.added} added to the archive"
)
for e in engine_elites:
    print(
        f"{e.short_id}  fitness={e.metrics['fitness']:.4f}  generation={e.lineage.generation}"
    )

await engine_storage.close()
engine_writer.close()

## The full mapping

| Part 1 (~150 lines) | Real GigaEvo (Part 2) |
|---|---|
| `llm()` helper | `MultiModelRouter` over `ChatOpenAI` — retries, multi-model routing, structured output |
| `evaluate()` + `signal.alarm` | stage DAG: `ValidateCodeStage` → `CallProgramFunction` (subprocess) → `CallValidatorFunction` → metrics stages |
| `Program` dataclass | `gigaevo.programs.Program` — Pydantic, `ProgramState` machine, full `Lineage` |
| `behavior_cell()` on code length | `BehaviorSpace` bins (default: 150 linear bins over fitness) |
| `archive` dict + `try_insert` | `MapElitesMultiIsland` + archive storage, size limits, tournament elite selection |
| skipping invalid children by hand | `StandardEvolutionAcceptor` — validity gate (`is_valid > 0`) + behavior keys + mutation context (§22) |
| `mutation_suggestion_stage()` | `MutationSuggestionStage` in the DAG → insights → `MutationContextStage` → mutation prompt |
| `mutation_operator()` + fenced-block parsing | `LLMMutationOperator.mutate_single` → structured `MutationSpec` |
| two nested `for` loops | `DagRunner` + `SteadyStateEvolutionEngine` (§22): async, parallel, backpressure, resume |

Everything §22 wired by hand — plus Redis storage, monitoring, snapshots/resume, and the
`gigaevo` CLI — is what `run.py` assembles from Hydra config. The whole notebook, as one
command (the LLM overrides are required — the config default points at OpenRouter Gemini;
`storage=disk` mirrors this notebook's Redis-free setup):

```bash
OPENAI_API_KEY=sk-gigaevo python run.py \
    problem.name=alphaevolve/packing_circles/n_26 \
    llm_base_url=http://localhost:8000/v1 \
    model_name=Qwen/Qwen3-235B-A22B-Instruct-2507 \
    storage=disk
```
